In [ ]:
# Set repo base BEFORE any config imports. This must be cell 0.
# The default config auto-detects the repo root from the REPO_ROOT env var
# but this repo is named 'analysis', so we override the base explicitly.
import os
os.environ['REPO_ROOT'] = os.environ.get('REPO_ROOT') or '/content/drive/MyDrive/cd-er-paradigm-choice'
print('[ok] REPO_ROOT =', os.environ['REPO_ROOT'])


# schema-poverty ablation Ditto Ablation ( SFT paradigm column of the schema-poverty ladder )

Companion to `05_schema_poverty_ablation_llm.ipynb` (LLM column, API-based).
This notebook fills the Ditto warm-start K=100 column of Table 18 in the paper
draft by fine-tuning Ditto at each rung/perturbation cell on the same target
records the LLM was evaluated on.

**Prerequisites:**
- Source K=full seed=42 checkpoint for Walmart-Amazon (trained via the source-training runner)
- Ditto AdamW patch applied (see `docs/methods/ditto_setup.md`)
- `results/schema_poverty_ablation/field_discriminativeness.csv` present

Cost: covered by existing Colab credits.

**Reduced scope (matches LLM notebook):**
- Method: Ditto warm-start K=100
- Targets: Amazon-Google, DBLP-ACM
- Rungs: 0-4 (dedup-skipping Amazon-Google rung 3)
- Perturbations at rungs 3-4: `none`, `glued`, `short_form` (drops `initials` and `shuffle`)
- Seeds: 42, 123, 456


## 1. Bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd $REPO_ROOT

In [ ]:
# Install dependencies (Ditto's stack + our config module)
!pip install -q transformers==4.36.2 sentencepiece pandas scikit-learn


In [ ]:
# Ensure Ditto repo is present and configured
import os
from pathlib import Path

DITTO_REPO = Path('$REPO_ROOT/ditto')
if not DITTO_REPO.exists():
    !git clone https://github.com/megagonlabs/ditto.git {DITTO_REPO}
print('[ok] Ditto repo at', DITTO_REPO)


In [ ]:
# Patch AdamW import (transformers 4.30+ moved AdamW to torch.optim)
# See docs/methods/ditto_setup.md
ditto_light = DITTO_REPO / 'ditto_light' / 'ditto.py'
text = ditto_light.read_text()
if 'from transformers import AdamW' in text:
    text = text.replace('from transformers import AdamW', 'from torch.optim import AdamW')
    ditto_light.write_text(text)
    print('[ok] AdamW import patched')
# Clear __pycache__ so the patch takes effect
!find {DITTO_REPO}/ditto_light -type d -name __pycache__ -exec rm -rf {{}} + 2>/dev/null || true
print('[ok] __pycache__ cleared')


In [ ]:
# GPU sanity check
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## 2. Config

In [ ]:
SOURCE = 'Structured/Walmart-Amazon'
# Optional override: set to the absolute .pt path if auto-detection fails.
# Leave as None to auto-detect via _find_source_pt.
SOURCE_PT_PATH = '$REPO_ROOT/models/checkpoints/Structured/Walmart-Amazon-kfull-s42/Structured/Walmart-Amazon-kfull-s42/model.pt'

TARGETS = ['Structured/Amazon-Google', 'Structured/DBLP-ACM', 'Textual/Abt-Buy']  # added Abt-Buy as 3rd target for regime diversity (textual product)
K = 100                    # target labels for warm-start fine-tune
SEEDS = [42, 123, 456]
RUNGS = [0, 1, 2, 3, 4]    # 5 rungs; dedup skips Amazon-Google rung 3 in-driver
PERTURBATIONS = ['none', 'glued', 'short_form']
PERTURBATION_RUNGS = {
    'none': RUNGS,
    'glued': [3, 4],
    'short_form': [3, 4],
}
SKIP_IF_DONE = True


n_cells = 0
for target in TARGETS:
    n_fields_by_target = {'Structured/Amazon-Google': 3, 'Structured/DBLP-ACM': 4, 'Textual/Abt-Buy': 3}
    for rung in RUNGS:
        for perturbation in PERTURBATIONS:
            if rung not in PERTURBATION_RUNGS[perturbation]:
                continue
            # simulate dedup: AG rung 3 duplicates rung 2, skip
            if target == 'Structured/Amazon-Google' and rung == 3:
                continue
            n_cells += len(SEEDS)
print(f'[info] Ablation matrix: {n_cells} total cells')


## 3. Verify source checkpoint

In [ ]:
from src.experiments.ditto_warmstart import _find_source_pt
from pathlib import Path as _P
if SOURCE_PT_PATH:
    src_pt = _P(SOURCE_PT_PATH)
    if src_pt.exists():
        print(f'[ok] source checkpoint (override): {src_pt}')
    else:
        print(f'[MISSING] override path does not exist: {src_pt}')
else:
    try:
        src_pt = _find_source_pt(SOURCE)
        print(f'[ok] source checkpoint (auto): {src_pt}')
    except FileNotFoundError as e:
        print(f'[MISSING] {e}')
        print('Fix options:')
        print(f'  (A) Set SOURCE_PT_PATH in cell 2 to the absolute .pt path, or')
        print(f'  (B) Run label-efficiency probe first to produce the source checkpoint.')


In [ ]:
!bash scripts/setup_colab.sh


## 4. Main loop

In [ ]:
# Driver call — the module handles masking, perturbation, Ditto training, metrics
from src.experiments.ditto_schema_poverty_ablation import run_ablation_cell
import time

start = time.time()
all_results = []
for target in TARGETS:
    for rung in RUNGS:
        for perturbation in PERTURBATIONS:
            if rung not in PERTURBATION_RUNGS[perturbation]:
                continue
            # dedup skip for AG rung 3 (field set identical to rung 2)
            if target == 'Structured/Amazon-Google' and rung == 3:
                print(f'  [dedup skip] AG rung 3 = rung 2 = {{title}}')
                continue
            for seed in SEEDS:
                print(f'\n=== target={target}  rung={rung}  perturb={perturbation}  seed={seed} ===')
                try:
                    r = run_ablation_cell(
                        SOURCE, target, rung, perturbation, seed,
                        k=K, skip_if_done=SKIP_IF_DONE,
                        source_pt_override=SOURCE_PT_PATH,
                    )
                    if r is not None:
                        all_results.append(r)
                except Exception as e:
                    print(f'  [ERROR] {e}')

elapsed_hr = (time.time() - start) / 3600
print(f'\n[done] {len(all_results)} cells complete  wall-clock: {elapsed_hr:.2f}h')


## 5. Aggregation

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import statistics as st

root = Path('$REPO_ROOT/results/runs/sparsity-ablation-sft')
files = sorted(root.rglob('metrics.json'))
print(f'Completed Ditto ablation cells: {len(files)}\n')

cells = defaultdict(list)
for f in files:
    d = json.loads(f.read_text())
    key = (d['target_dataset'], d['rung'], d['perturbation'])
    cells[key].append(d)

print(f"{'target':<30} {'r':>2} {'perturb':<12} {'seeds':<15} {'F1 (mean±sd)':<18}")
print('-' * 90)
for key in sorted(cells.keys()):
    target, rung, perturb = key
    ds = cells[key]
    seeds = sorted(d['seed'] for d in ds)
    f1s = [d.get('test_f1') for d in ds if d.get('test_f1') is not None]
    if not f1s:
        continue
    mean_f1 = st.mean(f1s)
    sd_f1 = st.stdev(f1s) if len(f1s) > 1 else 0.0
    seeds_str = ','.join(str(s) for s in seeds)
    tgt_short = target.replace('Structured/', '')
    print(f'{tgt_short:<30} {rung:>2} {perturb:<12} {seeds_str:<15} {mean_f1:.4f}±{sd_f1:.4f}')
